In [19]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [22]:
loan_df = pd.read_csv(
    r"C:\Users\henry\OneDrive\Desktop\LendingClub-Data-Pipeline\data\processed\clean_loan.csv"
)

In [31]:
fe_df = loan_df.copy()

In [45]:
date_cols = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "last_credit_pull_d"
]

for col in date_cols:
    fe_df[col] = pd.to_datetime(
        fe_df[col],
        errors="coerce"
    )
    
fe_df["issue_year"] = fe_df["issue_d"].dt.year
fe_df["issue_month"] = fe_df["issue_d"].dt.month
fe_df["issue_quarter"] = fe_df["issue_d"].dt.quarter


print(fe_df["issue_d"].head())
print(fe_df["issue_year"].head())
print(fe_df["issue_month"].head())
print(fe_df["issue_quarter"].head())


0   2018-12-01
1   2018-12-01
2   2018-12-01
3   2018-12-01
4   2018-12-01
Name: issue_d, dtype: datetime64[us]
0    2018
1    2018
2    2018
3    2018
4    2018
Name: issue_year, dtype: int32
0    12
1    12
2    12
3    12
4    12
Name: issue_month, dtype: int32
0    4
1    4
2    4
3    4
4    4
Name: issue_quarter, dtype: int32


In [62]:
fe_df["term"].head()

0    36
1    60
2    36
3    36
4    60
Name: term, dtype: int64

In [63]:
fe_df["term"] = fe_df["term"].str.replace(" months","",regex=False)


AttributeError: Can only use .str accessor with string values, not integer

In [67]:
fe_df["int_rate"]

0          13.56
1          18.94
2          17.97
3          18.94
4          16.14
           ...  
2260663    14.08
2260664    25.82
2260665    11.99
2260666    21.45
2260667    21.45
Name: int_rate, Length: 2260668, dtype: float64

In [ ]:
# Interest Rate

fe_df["int_rate"] = (
    fe_df["int_rate"].astype(str).str.replace("%","",regex= False).astype(float)
)

In [ ]:


fe_df["emp_length"]

0          1.0
1          1.0
2          6.0
3          1.0
4          1.0
          ... 
2260663    1.0
2260664    1.0
2260665    1.0
2260666    NaN
2260667    3.0
Name: emp_length, Length: 2260668, dtype: float64

In [112]:
# Employment Length

fe_df["emp_length"] = (
    fe_df["emp_length"].astype(str).str.extract(r"(\d)")
)

fe_df["emp_length"] = pd.to_numeric(
    fe_df["emp_length"],
    errors="coerce"
)


In [86]:
fe_df["emp_length"]

0          1.0
1          1.0
2          6.0
3          1.0
4          1.0
          ... 
2260663    1.0
2260664    1.0
2260665    1.0
2260666    NaN
2260667    3.0
Name: emp_length, Length: 2260668, dtype: float64

In [ ]:
# Loan to Income Ratio ⭐

fe_df["loan_income_ratio"] = (
    fe_df["loan_amnt"] /
    fe_df["annual_inc"]
)

In [ ]:
# Monthly Income

fe_df["monthly_income"] = (
    fe_df["annual_inc"] / 12
)

In [ ]:
# Installment Ratio

fe_df["installment_income_ratio"] = (
    fe_df["installment"]/
    fe_df["monthly_income"]
)


In [ ]:
# Credit Age


fe_df["credit_age"] = (
    fe_df["issue_year"] -
    fe_df["earliest_cr_line"].dt.year
)

In [ ]:
# Revolving Utilization Category


fe_df["revol_util_category"] = pd.cut(
    fe_df["revol_util"],
    bins=[0,30,60,100],
    labels=["Low","Medium","High"]
     
)

In [110]:
fe_df["revol_util_category"]

0             Low
1             Low
2             Low
3            High
4             Low
            ...  
2260663    Medium
2260664       Low
2260665      High
2260666    Medium
2260667    Medium
Name: revol_util_category, Length: 2260668, dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

In [118]:
fe_df["loan_category"] = pd.cut(
    fe_df["loan_amnt"],
    bins=[0,5000,15000,50000],
    labels=["Small",
        "Medium",
        "Large"]
)
fe_df["loan_category"]

0           Small
1           Large
2           Small
3           Small
4           Large
            ...  
2260663    Medium
2260664    Medium
2260665    Medium
2260666    Medium
2260667     Large
Name: loan_category, Length: 2260668, dtype: category
Categories (3, str): ['Small' < 'Medium' < 'Large']

In [120]:
fe_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,purpose,loan_status,emp_title,emp_length,home_ownership,annual_inc,verification_status,application_type,addr_state,zip_code,grade,sub_grade,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,acc_now_delinq,out_prncp,total_pymnt,total_rec_prncp,total_rec_int,recoveries,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,issue_d,total_rev_hi_lim,issue_year,issue_month,issue_quarter,loan_income_ratio,monthly_income,installment_income_ratio,credit_age,revol_util_category,loan_category
0,2500,2500,2500.0,36,13.56,84.92,debt_consolidation,Current,Chef,1.0,RENT,55000.0,Not Verified,Individual,NY,109xx,C,C1,18.24,0.0,2001-04-01,1.0,9.0,1.0,4341,10.3,34.0,1970-01-01,2386.02,167.02,113.98,53.04,0.0,2019-02-01,84.92,2019-02-01,2018-12-01,42000.0,2018,12,4,0.045455,4583.333333,0.018528,17.0,Low,Small
1,30000,30000,30000.0,60,18.94,777.23,debt_consolidation,Current,Postmaster,1.0,MORTGAGE,90000.0,Source Verified,Individual,LA,713xx,D,D2,26.52,0.0,1987-06-01,0.0,13.0,1.0,12315,24.2,44.0,1970-01-01,29387.75,1507.11,612.25,894.86,0.0,2019-02-01,777.23,2019-02-01,2018-12-01,50800.0,2018,12,4,0.333333,7500.000000,0.103631,31.0,Low,Large
2,5000,5000,5000.0,36,17.97,180.69,debt_consolidation,Current,Administrative,6.0,MORTGAGE,59280.0,Source Verified,Individual,MI,490xx,D,D1,10.51,0.0,2011-04-01,0.0,8.0,0.0,4599,19.1,13.0,1970-01-01,4787.21,353.89,212.79,141.10,0.0,2019-02-01,180.69,2019-02-01,2018-12-01,24100.0,2018,12,4,0.084345,4940.000000,0.036577,7.0,Low,Small
3,4000,4000,4000.0,36,18.94,146.51,debt_consolidation,Current,IT Supervisor,1.0,MORTGAGE,92000.0,Source Verified,Individual,WA,985xx,D,D2,16.74,0.0,2006-02-01,0.0,10.0,0.0,5468,78.1,13.0,1970-01-01,3831.93,286.71,168.07,118.64,0.0,2019-02-01,146.51,2019-02-01,2018-12-01,7000.0,2018,12,4,0.043478,7666.666667,0.019110,12.0,High,Small
4,30000,30000,30000.0,60,16.14,731.78,debt_consolidation,Current,Mechanic,1.0,MORTGAGE,57250.0,Not Verified,Individual,MD,212xx,C,C4,26.35,0.0,2000-12-01,0.0,12.0,0.0,829,3.6,26.0,1970-01-01,29339.02,1423.21,660.98,762.23,0.0,2019-02-01,731.78,2019-02-01,2018-12-01,23100.0,2018,12,4,0.524017,4770.833333,0.153386,18.0,Low,Large


In [123]:
fe_df.isnull().sum().sort_values()

loan_amnt                        0
funded_amnt                      0
funded_amnt_inv                  0
term                             0
int_rate                         0
installment                      0
purpose                          0
loan_status                      0
emp_title                        0
home_ownership                   0
annual_inc                       0
verification_status              0
addr_state                       0
application_type                 0
zip_code                         0
grade                            0
total_rec_prncp                  0
sub_grade                        0
dti                              0
delinq_2yrs                      0
inq_last_6mths                   0
open_acc                         0
pub_rec                          0
revol_bal                        0
total_rec_int                    0
revol_util                       0
total_acc                        0
out_prncp                        0
total_pymnt         

In [125]:
fe_df["emp_length"] = fe_df["emp_length"].fillna("Unknown")

In [129]:
fe_df["last_pymnt_d"].isnull().mean() * 100

np.float64(0.1073134135574087)

In [131]:
fe_df.describe().T

c:\Users\henry\OneDrive\Desktop\LendingClub-Data-Pipeline\.myvenv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\henry\OneDrive\Desktop\LendingClub-Data-Pipeline\.myvenv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,min,25%,50%,75%,max,std
loan_amnt,2260668.0,15046.931228,500.0,8000.0,12900.0,20000.0,40000.0,9190.245488
funded_amnt,2260668.0,15041.664057,500.0,8000.0,12875.0,20000.0,40000.0,9188.413022
funded_amnt_inv,2260668.0,15023.437624,0.0,8000.0,12800.0,20000.0,40000.0,9192.331807
term,2260668.0,42.910319,36.0,36.0,36.0,60.0,60.0,10.867161
int_rate,2260668.0,13.092913,5.31,9.49,12.62,15.99,30.99,4.832114
installment,2260668.0,445.807646,4.93,251.65,377.99,593.32,1719.83,267.173725
annual_inc,2260668.0,77992.405698,0.0,46000.0,65000.0,93000.0,110000000.0,112696.101198
dti,2260668.0,18.823452,-1.0,11.9,17.84,24.48,999.0,14.177986
delinq_2yrs,2260668.0,0.306875,0.0,0.0,0.0,0.0,58.0,0.867225
earliest_cr_line,2260639,1999-12-10 16:39:39.736349,1933-03-01 00:00:00,1995-11-01 00:00:00,2001-04-01 00:00:00,2005-05-01 00:00:00,2015-11-01 00:00:00,NaN


In [132]:
fe_df.to_csv(
    r"C:\Users\henry\OneDrive\Desktop\LendingClub-Data-Pipeline\data\processed\loan_feature_engineered.csv",
    index= False
)

In [133]:
print("Feature Engineering Completed Successfully.")

# 📌 New Features Created
# Feature	Purpose
# issue_year	Year-wise loan analysis
# issue_month	Monthly trend analysis
# issue_quarter	Quarterly reporting
# loan_income_ratio	Financial burden analysis
# monthly_income	Income analysis
# installment_income_ratio	EMI affordability
# credit_age	Credit history length
# revol_util_category	Credit utilization segmentation
# income_category	Income segmentation
# loan_category	Loan size segmentation

Feature Engineering Completed Successfully.
